In [31]:
import os
import csv
from dotenv import load_dotenv 
import pandas as pd

# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.
import nest_asyncio

#create vector store chromaDB
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

#ingest document into chromaDB
from llama_index.core import Document
from llama_index.core.text_splitter import TokenTextSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.ingestion import IngestionPipeline

#setting up query engine 
from llama_index.core.prompts import PromptTemplate
from llama_index.llms.together import TogetherLLM
from llama_index.core import VectorStoreIndex

In [3]:
# Load variables from .env file
load_dotenv()
nest_asyncio.apply()

In [36]:
# Access the keys
openai_key = os.getenv("OPENAI_API_KEY")
google_key = os.getenv("GOOGLE_API_KEY")
together_ai_key = os.getenv("TOGETHER_AI_API_TOKEN")

print("Keys loaded successfully")

Keys loaded successfully


### CREATE VECTOR STORE 

In [9]:
vector_store_name = "mini-llma-articles"
chroma_client = chromadb.PersistentClient(path = f"/Users/r31881/Desktop/AI/helloai/data/{vector_store_name}")
chroma_collection = chroma_client.get_or_create_collection(vector_store_name)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)


- DOWNLOAD THE DATASET
- THIS DATASET HAS 14 ARTICLES FROM TOWARDS AI BLOG 



In [13]:
#!curl -o /Users/r31881/Desktop/AI/helloai/data/mini-llama-articles.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv


- now read the data that we will store in the vector database

In [14]:
path_to_data = "/Users/r31881/Desktop/AI/helloai/data/mini-llama-articles.csv"

In [16]:
rows = []
#load the file as a json 
with open(path_to_data, mode = "r", encoding = "utf-8") as file:
    csv_reader = csv.reader(file)
    for index, row in enumerate(csv_reader):
        if index == 0:
            continue
        rows.append(row)

In [17]:
len(rows)

14

In [18]:
rows[:3]

[["Beyond GPT-4: What's New?",
  'LLM Variants and Meta\'s Open Source Before shedding light on four major trends, I\'d share the latest Meta\'s Llama 2 and Code Llama. Meta\'s Llama 2 represents a sophisticated evolution in LLMs. This suite spans models pretrained and fine-tuned across a parameter spectrum of 7 billion to 70 billion. A specialized derivative, Llama 2-Chat, has been engineered explicitly for dialogue-centric applications. Benchmarking revealed Llama 2\'s superior performance over most extant open-source chat models. Human-centric evaluations, focusing on safety and utility metrics, positioned Llama 2-Chat as a potential contender against proprietary, closed-source counterparts. The development trajectory of Llama 2 emphasized rigorous fine-tuning methodologies. Meta\'s transparent delineation of these processes aims to catalyze community-driven advancements in LLMs, underscoring a commitment to collaborative and responsible AI development. Code Llama is built on top of

### INGEST  DOCUMENTS INTO CHROMADB

- convert the data in rows list into document object so that the LLamaIndex framework can process them 


In [26]:
documents = [Document(text = row[1], metadata = {"title":row[0],"url":row[2],"source_name":row[3]}) for row in rows]

- Define the splitter object that split the text into segments with 512 tokens, 
- with a 128 overlap between the segments

In [27]:
text_splitter = TokenTextSplitter(
    separator = " ", chunk_size = 512, chunk_overlap = 128
)

- Create the pipeline to apply the transformation on each chunk. 
- and store the transformed text in the chroma vector store 
- document->chunk->embedding == node in llamaindex world
- IngestionPipeline is just a plan we have defined how this pipeline will be executed. documents->chunks->enbedding->nodes->vectorDB(chromaDB)

In [28]:
pipeline = IngestionPipeline(
    transformations = [
        text_splitter,
        HuggingFaceEmbedding(model_name = "BAAI/bge-small-en-v1.5") # or we can also use openAI embedding
    ],
    vector_store = vector_store
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8459.77it/s]


- now we will run the entire IngestionPipeline using the .run method 

In [29]:
nodes = pipeline.run(documents = documents, show_progress = True)

Generating embeddings: 100%|██████████| 108/108 [00:01<00:00, 55.81it/s]


In [ ]:
nodes = pipeline.run(documents=documents, show_progress=True)

### SETTING UP QUERY ENGINE

- Use the Together AI service to access the Llama chat model

In [32]:
llm = TogetherLLM(
    model = "meta-llama/Llama-4-Scout-17B-16E-Instruct",
    api_key = together_ai_key
)

- create index from vector store

In [33]:
index = VectorStoreIndex.from_vector_store(vector_store, embed_model = "local:BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15687.75it/s]


- define a query engine that is responsible for retrieving related pieces of text, 
- and using a LLM to formulate the final answer


In [37]:
query_engine = index.as_query_engine(llm=llm)

### TEST QUERY ENGINE 

In [38]:
res = query_engine.query("How many parameters LLaMA 2 has?")
print(res.response)
# 'Llama 2 is available in four different model sizes: 7 billion, 13 billion, 34 billion, and 70 billion parameters.'


AuthenticationError: Error code: 401 - {'id': 'otGoaku-2kFHot-a1fafd05fb8f8e0f', 'error': {'message': 'Invalid API key provided. You can find your API key at https://api.together.ai/settings/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}